# Cleanup Parquet files #
This jupyter notebook is to be run after running [create_parquet_files.ipynb](create_parquet_files.ipynb)

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

# 2021 Data #
Need to create: \
table with main census data, \
table with geogrpahic ids and names, \
table with characteristic ids and names, \
table with a list of columns (6 total (3 for population, and 3 for rates)), \
and a table that relates columns to characteristics.

In [ ]:
# -------------------------------------------------------------------
# 2021 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/2021_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            DGUID,
            CHARACTERISTIC_ID,
            C1_COUNT_TOTAL,
            "C2_COUNT_MEN+",
            "C3_COUNT_WOMEN+",
            C10_RATE_TOTAL,
            "C11_RATE_MEN+",
            "C12_RATE_WOMEN+"
        FROM '{input_file}'
    )
    TO 'parquet_data/2021_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            CHARACTERISTIC_ID,
            CHARACTERISTIC_NAME
        FROM '{input_file}'
        ORDER BY CHARACTERISTIC_ID
    )
    TO 'parquet_data/2021_characteristics.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# Geography lookup table
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            DGUID,
            GEO_LEVEL,
            GEO_NAME
        FROM '{input_file}'
        ORDER BY DGUID
    )
    TO 'parquet_data/2021_geographies.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# List of data columns
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS column_values_2021;")

con.execute("""
    CREATE TABLE column_values_2021 (
        column_id INTEGER PRIMARY KEY,
        column_name VARCHAR
    );
""")

con.execute("""
    INSERT INTO column_values_2021 VALUES
        (1, 'C1_COUNT_TOTAL'),
        (2, 'C2_COUNT_MEN+'),
        (3, 'C3_COUNT_WOMEN+'),
        (4, 'C10_RATE_TOTAL'),
        (5, 'C11_RATE_MEN+'),
        (6, 'C12_RATE_WOMEN+');
""")

# -------------------------------------------------------------------
# Characteristic -> column relationship
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS characteristic_columns_2021;")

con.execute("""
    CREATE TABLE characteristic_columns_2021 (
        CENSUS_YEAR INTEGER,
        CHARACTERISTIC_ID INTEGER,
        column_id INTEGER
    );
""")

columns = [
    "C1_COUNT_TOTAL",
    "C2_COUNT_MEN+",
    "C3_COUNT_WOMEN+",
    "C10_RATE_TOTAL",
    "C11_RATE_MEN+",
    "C12_RATE_WOMEN+"
]

for i, column in enumerate(columns, start=1):
    con.execute(f"""
        INSERT INTO characteristic_columns_2021
        SELECT DISTINCT
            CENSUS_YEAR,
            CHARACTERISTIC_ID,
            {i}
        FROM '{input_file}'
        WHERE "{column}" IS NOT NULL;
    """)

# -------------------------------------------------------------------
# Save lookup tables
# -------------------------------------------------------------------

con.execute("""
    COPY column_values_2021
    TO 'parquet_data/2021_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY characteristic_columns_2021
    TO 'parquet_data/2021_characteristic_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 2016 Data #
Need to create: \
table with main census data, \
table with geogrpahic ids and names, \
table with characteristic ids and names, \
table with a list of columns (3 total), \
and a table that relates columns to characteristics.

In [ ]:
# -------------------------------------------------------------------
# 2016 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/2016_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            GEO_CODE,
            CHARACTERISTIC_ID,
            COUNT_TOTAL,
            COUNT_MALE,
            COUNT_FEMALE
        FROM '{input_file}'
    )
    TO 'parquet_data/2016_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            CHARACTERISTIC_ID,
            CHARACTERISTIC_NAME
        FROM '{input_file}'
        ORDER BY CHARACTERISTIC_ID
    )
    TO 'parquet_data/2016_characteristics.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# Geography lookup table
# -------------------------------------------------------------------

con.execute(f"""
    COPY (
        SELECT DISTINCT
            CENSUS_YEAR,
            GEO_CODE,
            GEO_LEVEL,
            GEO_NAME
        FROM '{input_file}'
        ORDER BY GEO_CODE
    )
    TO 'parquet_data/2016_geographies.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# -------------------------------------------------------------------
# List of data columns
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS column_values_2016;")

con.execute("""
    CREATE TABLE column_values_2016 (
        column_id INTEGER PRIMARY KEY,
        column_name VARCHAR
    );
""")

con.execute("""
    INSERT INTO column_values_2016 VALUES
        (1, 'COUNT_TOTAL'),
        (2, 'COUNT_MALE'),
        (3, 'COUNT_FEMALE');
""")

# -------------------------------------------------------------------
# Characteristic -> column relationship
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS characteristic_columns_2016;")

con.execute("""
    CREATE TABLE characteristic_columns_2016 (
        CENSUS_YEAR INTEGER,
        CHARACTERISTIC_ID INTEGER,
        column_id INTEGER
    );
""")

columns = [
    "COUNT_TOTAL",
    "COUNT_MALE",
    "COUNT_FEMALE",
]

for i, column in enumerate(columns, start=1):
    con.execute(f"""
        INSERT INTO characteristic_columns_2016
        SELECT DISTINCT
            CENSUS_YEAR,
            CHARACTERISTIC_ID,
            {i}
        FROM '{input_file}'
        WHERE "{column}" IS NOT NULL;
    """)

# -------------------------------------------------------------------
# Save lookup tables
# -------------------------------------------------------------------

con.execute("""
    COPY column_values_2016
    TO 'parquet_data/2016_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY characteristic_columns_2016
    TO 'parquet_data/2016_characteristic_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 2011 Data #
Need to create: \
table with main census data, \
table with geogrpahic ids and names, \
table with characteristic ids and names, \
table with a list of columns (3 total), \
and a table that relates columns to characteristics.

In [ ]:
# -------------------------------------------------------------------
# 2011 Census / NHS
# -------------------------------------------------------------------

input_file1 = "temp_parquet_data/2011_raw_census_data.parquet"
input_file2 = "temp_parquet_data/2011_raw_nhs_data.parquet"
input_file3 = "temp_parquet_data/2011_raw_nhs_da.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_2011;")

con.execute("""
    CREATE TABLE DATA_2011 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE,
        COUNT_MALE DOUBLE,
        COUNT_FEMALE DOUBLE
    );
""")

# 2011 Census
con.execute(f"""
    INSERT INTO DATA_2011
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL,
        COUNT_MALE,
        COUNT_FEMALE
    FROM '{input_file1}';
""")

# 2011 NHS
con.execute(f"""
    INSERT INTO DATA_2011
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL,
        COUNT_MALE,
        COUNT_FEMALE
    FROM '{input_file2}';
""")

# -------------------------------------------------------------------
# 2011 NHS DA
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DA_NHS_DATA_2011;")

con.execute("""
    CREATE TABLE DA_NHS_DATA_2011 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute(f"""
    INSERT INTO DA_NHS_DATA_2011
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    FROM '{input_file3}';
""")

# -------------------------------------------------------------------
# List of data columns
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS column_values_2011;")

con.execute("""
    CREATE TABLE column_values_2011 (
        column_id INTEGER PRIMARY KEY,
        column_name VARCHAR
    );
""")

con.execute("""
    INSERT INTO column_values_2011 VALUES
        (1, 'COUNT_TOTAL'),
        (2, 'COUNT_MALE'),
        (3, 'COUNT_FEMALE');
""")

# -------------------------------------------------------------------
# Characteristic -> column relationship
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS characteristic_columns_2011;")

con.execute("""
    CREATE TABLE characteristic_columns_2011 (
        CHARACTERISTIC_ID INTEGER,
        column_id INTEGER
    );
""")

columns = [
    "COUNT_TOTAL",
    "COUNT_MALE",
    "COUNT_FEMALE",
]

for i, column in enumerate(columns, start=1):
    con.execute(f"""
        INSERT INTO characteristic_columns_2011
        SELECT DISTINCT
            CHARACTERISTIC_ID,
            {i}
        FROM read_parquet(['{input_file2}', '{input_file1}'])
        WHERE "{column}" IS NOT NULL;
    """)

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/2011_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/2011_characteristics.parquet",
    index=False
)

# -------------------------------------------------------------------
# NHS DA characteristics lookup table
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DA_CHAR_2011;")

con.execute("""
    CREATE TABLE DA_CHAR_2011 (
        CHARACTERISTIC_ID INTEGER,
        CHARACTERISTIC_NAME VARCHAR
    );
""")

con.execute(f"""
    INSERT INTO DA_CHAR_2011
    SELECT DISTINCT
        CHARACTERISTIC_ID,
        CHARACTERISTIC_NAME
    FROM '{input_file3}'
    ORDER BY CHARACTERISTIC_ID;
""")

# -------------------------------------------------------------------
# Geographic information table
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS GEO_2011;")

con.execute("""
    CREATE TABLE GEO_2011 (
        GEO_CODE VARCHAR,
        GEO_NAME VARCHAR
    );
""")

con.execute(f"""
    INSERT INTO GEO_2011
    SELECT DISTINCT
        GEO_CODE,
        GEO_NAME
    FROM read_parquet(['{input_file2}', '{input_file1}'])
    ORDER BY GEO_CODE;
""")

# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------

con.execute("""
    COPY DATA_2011
    TO 'parquet_data/2011_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY column_values_2011
    TO 'parquet_data/2011_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY characteristic_columns_2011
    TO 'parquet_data/2011_characteristic_columns.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY DA_NHS_DATA_2011
    TO 'parquet_data/2011_nhs_da_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY DA_CHAR_2011
    TO 'parquet_data/2011_nhs_da_characteristics.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY GEO_2011
    TO 'parquet_data/2011_geographies.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 2006 Data #
Need to create: \
table with main census data, \
table with characteristic ids and names, \
No need to create geographic table since the first jupyter notebook handled that. No need to create column table since there is only total count.

In [ ]:
# -------------------------------------------------------------------
# 2006 Census / NHS
# -------------------------------------------------------------------

input_file = "temp_parquet_data/2006_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_2006;")

con.execute("""
    CREATE TABLE DATA_2006 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        CENSUS_YEAR INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 2006 Census
con.execute(f"""
    INSERT INTO DATA_2006
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        CENSUS_YEAR,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/2006_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/2006_characteristics.parquet",
    index=False
)

# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------

con.execute("""
    COPY DATA_2006
    TO 'parquet_data/2006_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 2001 Data #
Need to create: \
table with main census data, \
table with characteristic ids and names, \
No need to create geographic table since the first jupyter notebook handled that. No need to create column table since there is only total count.

In [ ]:
# -------------------------------------------------------------------
# 2001 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/2001_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_2001;")

con.execute("""
    CREATE TABLE DATA_2001 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        CENSUS_YEAR INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 2001 Census
con.execute(f"""
    INSERT INTO DATA_2001
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        CENSUS_YEAR,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/2001_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/2001_characteristics.parquet",
    index=False
)

# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------

con.execute("""
    COPY DATA_2001
    TO 'parquet_data/2001_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 1996 Data #
Need to create: \
table with main census data, \
table with characteristic ids and names, \
No need to create geographic table since the first jupyter notebook handled that. No need to create column table since there is only total count.

In [ ]:
# -------------------------------------------------------------------
# 1996 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/1996_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1996;")

con.execute("""
    CREATE TABLE DATA_1996 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        CENSUS_YEAR INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 1996 Census
con.execute(f"""
    INSERT INTO DATA_1996
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        CENSUS_YEAR,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/1996_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/1996_characteristics.parquet",
    index=False
)

# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------

con.execute("""
    COPY DATA_1996
    TO 'parquet_data/1996_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 1991 Data #
Need to create: \
table with main census data, \
table with geogrpahic ids and names, \
table with characteristic ids and names, \
No need to create column table since there is only total count.

In [ ]:
# -------------------------------------------------------------------
# 1991 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/1991_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1991;")

con.execute("""
    CREATE TABLE DATA_1991 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 1991 Census
con.execute(f"""
    INSERT INTO DATA_1991
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/1991_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/1991_characteristics.parquet",
    index=False
)

# -------------------------------------------------------------------
# Geographic information table
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS GEO_1991;")

con.execute("""
    CREATE TABLE GEO_1991 (
        GEO_CODE VARCHAR,
        GEO_NAME VARCHAR
    );
""")

con.execute(f"""
    INSERT INTO GEO_1991
    SELECT DISTINCT
        GEO_CODE,
        GEO_NAME
    FROM '{input_file}'
    ORDER BY GEO_CODE;
""")

# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------

con.execute("""
    COPY DATA_1991
    TO 'parquet_data/1991_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY GEO_1991
    TO 'parquet_data/1991_geographies.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")


# 1986 Data #
Need to create: \
table with main census data, \
table with geogrpahic ids and names, \
table with characteristic ids and names, \
No need to create column table since there is only total count.

In [ ]:
# -------------------------------------------------------------------
# 1986 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/1986_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1986;")

con.execute("""
    CREATE TABLE DATA_1986 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 1986 Census
con.execute(f"""
    INSERT INTO DATA_1986
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/1986_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/1986_characteristics.parquet",
    index=False
)

# -------------------------------------------------------------------
# Geographic information table
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS GEO_1986;")

con.execute("""
    CREATE TABLE GEO_1986 (
        GEO_CODE VARCHAR,
        GEO_NAME VARCHAR
    );
""")

con.execute(f"""
    INSERT INTO GEO_1986
    SELECT DISTINCT
        GEO_CODE,
        GEO_NAME
    FROM '{input_file}'
    ORDER BY GEO_CODE;
""")

# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------

con.execute("""
    COPY DATA_1986
    TO 'parquet_data/1986_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY GEO_1986
    TO 'parquet_data/1986_geographies.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# 1981 Data #
Need to create: \
table with main census data, \
table with geogrpahic ids and names, \
table with characteristic ids and names, \
No need to create column table since there is only total count.

In [ ]:
# -------------------------------------------------------------------
# 1981 Census
# -------------------------------------------------------------------

input_file = "temp_parquet_data/1981_raw_census_data.parquet"

# -------------------------------------------------------------------
# Main data
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1981;")

con.execute("""
    CREATE TABLE DATA_1981 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

# 1981 Census
con.execute(f"""
    INSERT INTO DATA_1981
    SELECT DISTINCT
        GEO_CODE,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    FROM '{input_file}';
""")

# -------------------------------------------------------------------
# Characteristics lookup table
# -------------------------------------------------------------------

df = pd.read_csv(
    "characteristics/1981_characteristics.csv",
    usecols=["CHARACTERISTIC_ID", "CHARACTERISTIC_NAME"]
)

df.to_parquet(
    "parquet_data/1981_characteristics.parquet",
    index=False
)

# -------------------------------------------------------------------
# Geographic information table
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS GEO_1981;")

con.execute("""
    CREATE TABLE GEO_1981 (
        GEO_CODE VARCHAR,
        GEO_NAME VARCHAR
    );
""")

con.execute(f"""
    INSERT INTO GEO_1981
    SELECT DISTINCT
        GEO_CODE,
        GEO_NAME
    FROM '{input_file}'
    ORDER BY GEO_CODE;
""")

# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------

con.execute("""
    COPY DATA_1981
    TO 'parquet_data/1981_census_data.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("""
    COPY GEO_1981
    TO 'parquet_data/1981_geographies.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# Tables for 1976 to 1871 Census #

In [ ]:
# Canada's borders remained unchanged since 1949. So, I will use the
# 1981 Census boundary vector tiles for 1951 to 1976. The Geo_Code's
# must match.

# -------------------------------------------------------------------
# 1976 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1976;")

con.execute("""
    CREATE TABLE DATA_1976 (
        GEO_CODE VARCHAR,
        GEO_NAME VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""\
    INSERT INTO DATA_1976 (
        GEO_CODE,
        GEO_NAME,
        CHARACTERISTIC_ID,
        COUNT_TOTAL
    )
    VALUES
        ('35', 'Ontario', 1, 8264465),
        ('24', 'Quebec', 1, 6234445),
        ('59', 'British Columbia', 1, 2466608),
        ('48', 'Alberta', 1, 1838037),
        ('46', 'Manitoba', 1, 1021506),
        ('47', 'Saskatchewan', 1, 921323),
        ('12', 'Nova Scotia', 1, 828571),
        ('13', 'New Brunswick', 1, 677250),
        ('10', 'Newfoundland and Labrador', 1, 557725),
        ('11', 'Prince Edward Island', 1, 118229),
        ('61', 'Northwest Territories', 1, 42609),
        ('60', 'Yukon', 1, 21836),
        ('00', 'Canada', 1, 22992604);
""")

# -------------------------------------------------------------------
# 1971 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1971;")

con.execute("""
    CREATE TABLE DATA_1971 (
        GEO_CODE VARCHAR,
        GEO_NAME VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1971 VALUES
        ('35', 'Ontario', 1, 7703106),
        ('24', 'Quebec', 1, 6027764),
        ('59', 'British Columbia', 1, 2184621),
        ('48', 'Alberta', 1, 1627874),
        ('46', 'Manitoba', 1, 988247),
        ('47', 'Saskatchewan', 1, 926242),
        ('12', 'Nova Scotia', 1, 788960),
        ('13', 'New Brunswick', 1, 634557),
        ('10', 'Newfoundland and Labrador', 1, 522104),
        ('11', 'Prince Edward Island', 1, 111641),
        ('61', 'Northwest Territories', 1, 34807),
        ('60', 'Yukon', 1, 18388),
        ('00', 'Canada', 1, 21568311);
""")


# -------------------------------------------------------------------
# 1961 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1961;")

con.execute("""
    CREATE TABLE DATA_1961 (
        GEO_CODE VARCHAR,
        GEO_NAME VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1961 VALUES
        ('35', 'Ontario', 1, 6236092),
        ('24', 'Quebec', 1, 5259211),
        ('59', 'British Columbia', 1, 1629082),
        ('48', 'Alberta', 1, 1331944),
        ('46', 'Manitoba', 1, 921686),
        ('47', 'Saskatchewan', 1, 925181),
        ('12', 'Nova Scotia', 1, 737007),
        ('13', 'New Brunswick', 1, 597936),
        ('10', 'Newfoundland and Labrador', 1, 457853),
        ('11', 'Prince Edward Island', 1, 104629),
        ('61', 'Northwest Territories', 1, 22998),
        ('60', 'Yukon', 1, 14628),
        ('00', 'Canada', 1, 18238247);
""")


# -------------------------------------------------------------------
# 1951 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1951;")

con.execute("""
    CREATE TABLE DATA_1951 (
        GEO_CODE VARCHAR,
        GEO_NAME VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1951 VALUES
        ('35', 'Ontario', 1, 4597542),
        ('24', 'Quebec', 1, 4055681),
        ('59', 'British Columbia', 1, 1165210),
        ('48', 'Alberta', 1, 939501),
        ('46', 'Manitoba', 1, 776541),
        ('47', 'Saskatchewan', 1, 831728),
        ('12', 'Nova Scotia', 1, 642584),
        ('13', 'New Brunswick', 1, 515697),
        ('10', 'Newfoundland and Labrador', 1, 361416),
        ('11', 'Prince Edward Island', 1, 98429),
        ('61', 'Northwest Territories', 1, 16004),
        ('60', 'Yukon', 1, 9096),
        ('00', 'Canada', 1, 14009429);
""")

# -------------------------------------------------------------------
# 1941 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1941;")

con.execute("""
    CREATE TABLE DATA_1941 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1941 VALUES
        ('Ontario', 1, 3787655),
        ('Quebec', 1, 3331882),
        ('British Columbia', 1, 817861),
        ('Alberta', 1, 796169),
        ('Manitoba', 1, 729744),
        ('Saskatchewan', 1, 895992),
        ('Nova Scotia', 1, 577962),
        ('New Brunswick', 1, 457401),
        ('Prince Edward Island', 1, 95047),
        ('Northwest Territories', 1, 12028),
        ('Yukon', 1, 4914),
        ('Canada', 1, 11506655);
""")

# -------------------------------------------------------------------
# 1931 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1931;")

con.execute("""
    CREATE TABLE DATA_1931 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1931 VALUES
        ('Ontario', 1, 3431683),
        ('Quebec', 1, 2874662),
        ('British Columbia', 1, 694263),
        ('Alberta', 1, 731605),
        ('Manitoba', 1, 700139),
        ('Saskatchewan', 1, 921785),
        ('Nova Scotia', 1, 512846),
        ('New Brunswick', 1, 408219),
        ('Prince Edward Island', 1, 88038),
        ('Northwest Territories', 1, 9316),
        ('Yukon', 1, 4230),
        ('Canada', 1, 10376786);
""")

# -------------------------------------------------------------------
# 1921 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1921;")

con.execute("""
    CREATE TABLE DATA_1921 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1921 VALUES
        ('Ontario', 1, 2933622),
        ('Quebec', 1, 2361199),
        ('British Columbia', 1, 524582),
        ('Alberta', 1, 588454),
        ('Manitoba', 1, 610118),
        ('Saskatchewan', 1, 757510),
        ('Nova Scotia', 1, 523837),
        ('New Brunswick', 1, 387876),
        ('Prince Edward Island', 1, 88615),
        ('Northwest Territories', 1, 7988),
        ('Yukon', 1, 4157),
        ('Canada', 1, 8787949);
""")

# -------------------------------------------------------------------
# 1911 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1911;")

con.execute("""
    CREATE TABLE DATA_1911 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1911 VALUES
        ('Ontario', 1, 2527292),
        ('Quebec', 1, 2005776),
        ('British Columbia', 1, 392480),
        ('Alberta', 1, 374295),
        ('Manitoba', 1, 461394),
        ('Saskatchewan', 1, 492432),
        ('Nova Scotia', 1, 492338),
        ('New Brunswick', 1, 351889),
        ('Prince Edward Island', 1, 93728),
        ('Northwest Territories', 1, 17196),
        ('Yukon', 1, 8512),
        ('Canada', 1, 7206643);
""")

# -------------------------------------------------------------------
# 1901 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1901;")

con.execute("""
    CREATE TABLE DATA_1901 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1901 VALUES
        ('Ontario', 1, 2182397),
        ('Quebec', 1, 1648398),
        ('British Columbia', 1, 178657),
        ('Alberta', 1, 73022),
        ('Manitoba', 1, 255211),
        ('Saskatchewan', 1, 91497),
        ('Nova Scotia', 1, 459574),
        ('New Brunswick', 1, 331120),
        ('Prince Edward Island', 1, 103259),
        ('Northwest Territories', 1, 20000),
        ('Yukon', 1, 27000),
        ('Canada', 1, 5371315);
""")

# -------------------------------------------------------------------
# 1891 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1891;")

con.execute("""
    CREATE TABLE DATA_1891 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1891 VALUES
        ('Ontario', 1, 2114321),
        ('Quebec', 1, 1488535),
        ('British Columbia', 1, 98173),
        ('Manitoba', 1, 152506),
        ('Nova Scotia', 1, 450396),
        ('New Brunswick', 1, 321263),
        ('Prince Edward Island', 1, 109078),
        ('Northwest Territories', 1, 98967),
        ('Canada', 1, 4833239);
""")

# -------------------------------------------------------------------
# 1881 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1881;")

con.execute("""
    CREATE TABLE DATA_1881 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1881 VALUES
        ('Ontario', 1, 1924271),
        ('Quebec', 1, 1358653),
        ('British Columbia', 1, 48886),
        ('Manitoba', 1, 64824),
        ('Nova Scotia', 1, 440558),
        ('New Brunswick', 1, 321251),
        ('Prince Edward Island', 1, 108911),
        ('Northwest Territories', 1, 10973),
        ('Canada', 1, 4324810);
""")

# -------------------------------------------------------------------
# 1871 Census
# -------------------------------------------------------------------

con.execute("DROP TABLE IF EXISTS DATA_1871;")

con.execute("""
    CREATE TABLE DATA_1871 (
        GEO_CODE VARCHAR,
        CHARACTERISTIC_ID INTEGER,
        COUNT_TOTAL DOUBLE
    );
""")

con.execute("""
    INSERT INTO DATA_1871 VALUES
        ('Ontario', 1, 1620851),
        ('Quebec', 1, 1191516),
        ('Nova Scotia', 1, 387800),
        ('New Brunswick', 1, 285594),
        ('Prince Edward Island', 1, 94021),
        ('Manitoba', 1, 25228),
        ('British Columbia', 1, 36247),
        ('Northwest Territories', 1, 48000),
        ('Canada', 1, 3689257);
""")


# -------------------------------------------------------------------
# Save everything
# -------------------------------------------------------------------
def save_file(census_year):
    con.execute(f"""
        COPY DATA_{census_year}
        TO 'parquet_data/{census_year}_census_data.parquet'
        (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

CENSUS_YEARS = [
    1871, 1881, 1891, 1901, 1911, 1921, 1931, 1941,
    1951, 1961, 1971, 1976
]

for year in CENSUS_YEARS:
    save_file(year)


# Run this to close the connection #

In [ ]:
con.close()